# Cross-sensor contrastive BEATs

Two pretrained BEATs encoders align audio and acceleration from the same surface, without family labels.
Each batch contains one setup. Random windows need no temporal alignment, including in LMT108.
Pre-training and logistic regression use two setups; the third is held out entirely.
Compare against frozen BEATs with the same classifier.

Run this notebook alone with Python 3.12+ and a GPU (16 GB recommended). All experiment code is here;
packages, data and the official BEATs implementation are downloaded automatically. The data archive
supplies only two recording tables and two waveform arrays; no repository Python files are imported.
`QUICK = True` uses one window per recording and 30 updates. Lower `BATCH` to 64 if memory runs out.
Results go to `analysis/classification/cross_sensor/` (`cross_sensor_quick/` for checks).
Finished stages are reused; start a fresh results folder after changing settings.


In [ ]:
QUICK = False

DRIVE_FILE_ID = "1NmRCDqvGvmNwQGC53IzTu9jJRaI8Newk"

SETUPS = ["cluster", "lmt108", "sens3"]
FAMILIES = ["wood", "metal", "textile", "polymer"]
INPUTS = {"audio": ["audio"], "acceleration": ["accel"], "audio + acceleration": ["audio", "accel"]}
N_UPDATES = 2000
BATCH = 96
TEMPERATURE = 0.1
EMBED_DIM = 256
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WAVE_RMS = 0.1

## Packages, data and BEATs

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy",
    "torch": "torch",
    "torchaudio": "torchaudio",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    import site

    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
DATA_FILES = [
    "analysis/classification/waveforms.npz",
    "analysis/classification/sens3_waveforms.npz",
    "analysis/signal_processing/recordings.csv",
    "analysis/classification/sens3_recordings.csv",
]


def download_from_drive(file_id, path):
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    partial = path.with_suffix(".part")
    done = partial.stat().st_size if partial.exists() else 0
    request = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    with urllib.request.urlopen(request) as response, open(partial, "ab" if done else "wb") as out:
        if done and response.status != 206:
            out.truncate(0)
            done = 0
        assert "html" not in response.headers.get("Content-Type", ""), "Drive sent a web page: is the file shared?"
        total = done + int(response.headers.get("Content-Length", 0))
        while chunk := response.read(1 << 22):
            out.write(chunk)
            done += len(chunk)
            print(f"\r{done / 2**30:.2f} / {total / 2**30:.2f} GB", end="", flush=True)
    print()
    partial.rename(path)


if not all((ROOT / name).exists() for name in DATA_FILES):
    archive_path = ROOT / "data.tar"
    if not archive_path.exists():
        download_from_drive(DRIVE_FILE_ID, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(ROOT, members=[archive.getmember(name) for name in DATA_FILES], filter="data")
    archive_path.unlink()
print("data ready in", ROOT)

BEATS_DIR = ROOT / "models/beats"
BEATS_DIR.mkdir(parents=True, exist_ok=True)
BEATS_CODE = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
BEATS_WEIGHTS = "https://huggingface.co/alibabasglab/beats/resolve/main/BEATs_iter3_plus_AS2M.pt"
for name in ["BEATs.py", "backbone.py", "modules.py"]:
    if not (BEATS_DIR / name).exists():
        urllib.request.urlretrieve(BEATS_CODE + name, BEATS_DIR / name)
if not (BEATS_DIR / "BEATs_iter3_plus_AS2M.pt").exists():
    print("downloading BEATs weights (360 MB)...")
    urllib.request.urlretrieve(BEATS_WEIGHTS, BEATS_DIR / "BEATs_iter3_plus_AS2M.pt")
print("BEATs ready in", BEATS_DIR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchaudio
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from torch import nn

sys.path.insert(0, str(BEATS_DIR))
from BEATs import BEATs, BEATsConfig  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 2**30:.0f} GB")
else:
    print("no GPU: far too slow")

AMP_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

RESULTS = ROOT / "analysis/classification" / ("cross_sensor_quick" if QUICK else "cross_sensor")
RESULTS.mkdir(parents=True, exist_ok=True)

## Evaluation

Balanced accuracy, chance 0.25; 95% intervals bootstrap whole surfaces within families.
Training weights balance each (setup, family) group.


In [ ]:
def group_weights(recordings):
    weights = 1 / recordings.groupby(["dataset", "family"]).family.transform("size").to_numpy()
    return weights / weights.mean()


def fit_and_predict(train_features, train_recordings, test_features):
    scaler = StandardScaler().fit(train_features)
    model = LogisticRegression(max_iter=3000)
    model.fit(scaler.transform(train_features), train_recordings.family, sample_weight=group_weights(train_recordings))
    return model.predict(scaler.transform(test_features))


def mean_recall(confusion):
    return np.mean(np.diag(confusion) / confusion.sum(axis=1))


def score(recordings, predicted):
    confusions = {}
    for surface, rows in recordings.groupby("surface").indices.items():
        confusions[surface] = confusion_matrix(recordings.family.iloc[rows], predicted[rows], labels=FAMILIES)
    family_of_surface = recordings.groupby("surface").family.first()
    rng = np.random.default_rng(42)
    draws = []
    for _ in range(2000):
        resampled = sum(
            confusions[surface]
            for family in FAMILIES
            for surface in rng.choice(
                family_of_surface.index[family_of_surface == family], (family_of_surface == family).sum()
            )
        )
        draws.append(mean_recall(resampled))
    low, high = np.percentile(draws, [2.5, 97.5])
    return {"balanced accuracy": mean_recall(sum(confusions.values())), "low": low, "high": high}

## Data

Unit-RMS windows: audio at 16 kHz, acceleration resampled from 8 to 16 kHz for BEATs.


In [ ]:
main = pd.read_csv(ROOT / "analysis/signal_processing/recordings.csv", dtype={"surface": str})
sens3 = pd.read_csv(ROOT / "analysis/classification/sens3_recordings.csv", dtype={"surface": str})
sens3["dataset"] = "sens3"
recordings = pd.concat([main, sens3], ignore_index=True)


def load_waves(name):
    cache = np.load(ROOT / "analysis/classification" / name)
    recording_of_window = cache["recording"]
    keep = np.ones(len(recording_of_window), bool)
    if QUICK:
        keep[1:] = recording_of_window[1:] != recording_of_window[:-1]
    accel_8k = cache["accel"][keep]
    accel = torch.empty((len(accel_8k), 2 * accel_8k.shape[1]), dtype=torch.float16)
    for start in range(0, len(accel_8k), 4096):
        piece = torch.from_numpy(accel_8k[start : start + 4096].astype(np.float32))
        accel[start : start + 4096] = torchaudio.functional.resample(piece, 8000, 16000).half()
    return {"audio": torch.from_numpy(cache["audio"][keep]), "accel": accel}, recording_of_window[keep]


main_waves, main_recording_of_window = load_waves("waveforms.npz")
sens3_waves, sens3_recording_of_window = load_waves("sens3_waveforms.npz")
waves = {name: torch.cat([main_waves[name], sens3_waves[name]]) for name in ["audio", "accel"]}
recording_of_window = np.concatenate([main_recording_of_window, sens3_recording_of_window + len(main)])
del main_waves, sens3_waves

first_window = np.searchsorted(recording_of_window, np.arange(len(recordings)))
n_windows = np.bincount(recording_of_window, minlength=len(recordings))
surface_index = pd.factorize(recordings.dataset + " " + recordings.surface)[0]
print(recordings.groupby("dataset").agg(recordings=("surface", "size"), surfaces=("surface", "nunique")))
print(len(recording_of_window), "windows")

## Model and contrastive pre-training

In [ ]:
def load_beats():
    checkpoint = torch.load(BEATS_DIR / "BEATs_iter3_plus_AS2M.pt", map_location="cpu", weights_only=True)
    model = BEATs(BEATsConfig(checkpoint["cfg"]))
    model.load_state_dict(checkpoint["model"])
    preprocess = model.preprocess

    def preprocess_in_float32(source, **kwargs):
        with torch.autocast("cuda", enabled=False):
            return preprocess(source.float(), **kwargs)

    model.preprocess = preprocess_in_float32
    return model.to(DEVICE)


def beats_embedding(model, x):
    features, _ = model.extract_features(x.float() * WAVE_RMS)
    return features.mean(dim=1)


class CrossSensor(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoders = nn.ModuleDict({name: load_beats() for name in ["audio", "accel"]})
        width = self.encoders["audio"].cfg.encoder_embed_dim
        self.heads = nn.ModuleDict({name: nn.Linear(width, EMBED_DIM) for name in ["audio", "accel"]})

    def forward(self, name, x):
        return nn.functional.normalize(self.heads[name](beats_embedding(self.encoders[name], x)), dim=1)


def contrastive_loss(audio, accel, surfaces):
    similarity = audio @ accel.T / TEMPERATURE
    positive = torch.as_tensor(surfaces[:, None] == surfaces[None, :], device=DEVICE).float()
    target = positive / positive.sum(dim=1, keepdim=True)
    from_audio = -(target * similarity.log_softmax(dim=1)).sum(dim=1).mean()
    from_accel = -(target.T * similarity.T.log_softmax(dim=1)).sum(dim=1).mean()
    return (from_audio + from_accel) / 2


def pretrain(training_setups, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CrossSensor().to(DEVICE)
    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoders.parameters(), "lr": ENCODER_LR},
            {"params": model.heads.parameters(), "lr": HEAD_LR},
        ],
        weight_decay=0.01,
    )
    n_updates = 30 if QUICK else N_UPDATES
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[ENCODER_LR, HEAD_LR], total_steps=n_updates, pct_start=0.1
    )
    scaler = torch.amp.GradScaler(enabled=torch.float16 == AMP_DTYPE)
    recordings_of_setup = {setup: np.flatnonzero(recordings.dataset == setup) for setup in training_setups}

    model.train()
    for update in range(n_updates):
        recs = rng.choice(recordings_of_setup[rng.choice(training_setups)], BATCH, replace=False)
        audio_windows = first_window[recs] + rng.integers(0, n_windows[recs])
        accel_windows = first_window[recs] + rng.integers(0, n_windows[recs])
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            audio = model("audio", waves["audio"][audio_windows].to(DEVICE)).float()
            accel = model("accel", waves["accel"][accel_windows].to(DEVICE)).float()
        loss = contrastive_loss(audio, accel, surface_index[recs])
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        schedule.step()
        if update % 200 == 0:
            print(f"update {update}: loss {loss.item():.3f}", flush=True)
    return model


@torch.no_grad()
def recording_embeddings(embed):
    out = {}
    for name in ["audio", "accel"]:
        parts = []
        for start in range(0, len(recording_of_window), 256):
            x = waves[name][start : start + 256].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
                parts.append(embed(name, x).float().cpu())
        out[name] = pd.DataFrame(torch.cat(parts).numpy()).groupby(recording_of_window).mean().to_numpy()
    return out

## Embeddings

Frozen baseline, then pre-training without each test setup. Completed embeddings are cached.


In [ ]:
def cached(name, make_embed):
    path = RESULTS / f"embeddings_{name}.npz"
    if not path.exists():
        embed = make_embed()
        np.savez(path, **recording_embeddings(embed))
        del embed
        torch.cuda.empty_cache()
    return dict(np.load(path))


def frozen_beats():
    encoder = load_beats().eval()
    return lambda name, x: beats_embedding(encoder, x)


def pretrained_without(held_out):
    model = pretrain([setup for setup in SETUPS if setup != held_out]).eval()
    return lambda name, x: model(name, x)


embeddings = {("frozen BEATs", None): cached("frozen_beats", frozen_beats)}
for held_out in SETUPS:
    print("pre-training without", held_out, flush=True)
    embeddings[("cross-sensor BEATs", held_out)] = cached(
        f"cross_sensor_without_{held_out}", lambda held_out=held_out: pretrained_without(held_out)
    )

## Results

Logistic regression trained on two setups, evaluated on the third.


In [ ]:
rows = []
for held_out in SETUPS:
    test = (recordings.dataset == held_out).to_numpy()
    for model_name in ["frozen BEATs", "cross-sensor BEATs"]:
        signals = embeddings[(model_name, None if model_name == "frozen BEATs" else held_out)]
        for input_name, names in INPUTS.items():
            features = np.concatenate([signals[name] for name in names], axis=1)
            predicted = fit_and_predict(features[~test], recordings[~test], features[test])
            rows.append(
                {
                    "tested on": held_out,
                    "model": model_name,
                    "input": input_name,
                    **score(recordings[test].reset_index(drop=True), predicted),
                }
            )
            print(held_out, model_name, input_name, f"{rows[-1]['balanced accuracy']:.2f}", flush=True)

scores = pd.DataFrame(rows)
scores.to_csv(RESULTS / "cross_sensor_scores.csv", index=False)
scores.pivot_table(index=["tested on", "input"], columns="model", values="balanced accuracy").round(2)

In [ ]:
fig, axes = plt.subplots(1, len(SETUPS), figsize=(15, 4), sharey=True)
for ax, held_out in zip(axes, SETUPS, strict=True):
    for k, model_name in enumerate(["frozen BEATs", "cross-sensor BEATs"]):
        bars = scores[(scores["tested on"] == held_out) & (scores.model == model_name)].set_index("input")
        x = np.arange(len(INPUTS)) + (k - 0.5) * 0.4
        ax.bar(x, bars["balanced accuracy"], 0.4, color=f"C{k}", label=model_name)
        ax.errorbar(
            x,
            bars["balanced accuracy"],
            yerr=[bars["balanced accuracy"] - bars.low, bars.high - bars["balanced accuracy"]],
            fmt="none",
            ecolor="black",
            capsize=2,
        )
    ax.axhline(0.25, color="black", linestyle=":", linewidth=1)
    ax.set_xticks(range(len(INPUTS)), list(INPUTS), rotation=20)
    ax.set(title=f"tested on {held_out}", ylim=(0, 1))
axes[0].set_ylabel("balanced accuracy, 95% interval\n(dotted line: chance)")
axes[-1].legend(bbox_to_anchor=(1, 1), loc="upper left")
fig.suptitle("Cross-sensor contrastive pre-training, trained on two setups, tested on the third")
fig.savefig(RESULTS / "cross_sensor.png", dpi=130, bbox_inches="tight")
plt.show()